# Fusion Trust Model Training
Train a LightGBM-based Fusion Trust Model using the synthetic dataset.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import joblib

try:
    import lightgbm as lgb
except ImportError:
    !pip install lightgbm -q
    import lightgbm as lgb


## Load Dataset

In [ ]:
df = pd.read_csv("fusion_trust_model_synthetic_10000.csv")
df.head()


In [ ]:
FEATURES = [
    "sep_score",
    "semantic_entropy",
    "kernel_language_entropy",
    "nli_agreement",
    "retrieval_confidence",
    "symbolic_verification",
    "temporal_validation",
    "cross_model_agreement",
    "critic_score",
    "calibration_factor"
]

TARGET = "fusion_trust_score"

X = df[FEATURES]
y = df[TARGET]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)


## Train LightGBM Regressor

In [ ]:
model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="l2",
    callbacks=[
        lgb.early_stopping(30),
        lgb.log_evaluation(20)
    ]
)

joblib.dump(model, "fusion_trust_model.pkl")
print("Model saved as fusion_trust_model.pkl")


## Evaluation

In [ ]:
pred = model.predict(X_test)

print("MSE :", mean_squared_error(y_test, pred))
print("MAE :", mean_absolute_error(y_test, pred))
print("R2  :", r2_score(y_test, pred))


## Feature Importance

In [ ]:
importance = model.feature_importances_

plt.figure(figsize=(8,5))
plt.barh(FEATURES, importance)
plt.title("Feature Importance")
plt.show()


## Prediction Example

In [ ]:
sample = X_test.iloc[[0]]
score = model.predict(sample)[0]

print("Predicted Trust Score:", score)

if score >= 0.85:
    print("Risk: Low")
elif score >= 0.60:
    print("Risk: Medium")
else:
    print("Risk: High")


## Load Saved Model

In [ ]:
loaded_model = joblib.load("fusion_trust_model.pkl")
loaded_model.predict(X_test.iloc[:5])
